# BCG GenAI Consulting – Financial Data Analysis
## GFC AI Chatbot Project: Task 1 Deliverable
**Analyst:** Harsh  
**Manager:** Aisha, Senior Data Scientist, BCG GenAI Consulting Team  
**Data Source:** SEC EDGAR 10-K Filings — Microsoft (MSFT), Tesla (TSLA), Apple (AAPL) — FY2023 to FY2025  
**All figures in USD Millions**

---
## 1. Understanding 10-K & 10-Q Reports (SEC Framework)

| Item | Section | What We Extract |
|------|---------|----------------|
| Item 1 | Business | Company overview, products, markets |
| Item 1A | Risk Factors | Key business risks (ordered by importance) |
| Item 7 | MD&A | Management's perspective on results, liquidity, trends |
| Item 8 | Financial Statements | Income statement, balance sheet, cash flows (GAAP audited) |

**Key reliability checks:**
- CEO & CFO certify accuracy of all filings (Sections 302 & 906)
- Financial statements follow GAAP
- Independent auditor review — unqualified opinion confirmed for all three companies
- GAAP figures used exclusively (non-GAAP excluded)

---
## 2. Data Sources

| Company | Ticker | Source |
|---------|--------|--------|
| Microsoft | MSFT | https://www.microsoft.com/investor/reports/ar25/index.html |
| Tesla | TSLA | https://stocklight.com/stocks/us/nasdaq-tsla/tesla/annual-reports/nasdaq-tsla-2026-10K-26574326.pdf |
| Apple | AAPL | https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm |

In [1]:
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


---
## 3. Load Data from Excel

In [2]:
# Load the data from the Excel spreadsheet
file_path = 'BCG_Financial_Data_Harsh.xlsx'
df = pd.read_excel(file_path, sheet_name='Extracted Data')

# Keep only the raw financial columns
df = df[['Company Name','Ticker','Fiscal Year','Fiscal Year End',
         'Total Revenue','Net Income','Total Assets',
         'Total Liabilities','Cash Flow from Operating Activities']].copy()

print("Shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
df

Shape: (9, 9)

Missing values: 0


,Company Name,Ticker,Fiscal Year,Fiscal Year End,Total Revenue,Net Income,Total Assets,Total Liabilities,Cash Flow from Operating Activities
0,Microsoft,MSFT,2023,2023-06-30,211915,72361,411976,205753,87582
1,Microsoft,MSFT,2024,2024-06-30,245122,88136,512163,243686,118548
2,Microsoft,MSFT,2025,2025-06-30,281724,101832,619003,275524,136162
3,Tesla,TSLA,2023,2023-12-31,96773,14974,106618,43009,13256
4,Tesla,TSLA,2024,2024-12-31,97690,7153,122070,48390,14923
5,Tesla,TSLA,2025,2025-12-31,94827,3855,137806,54941,14747
6,Apple,AAPL,2023,2023-09-30,383285,96995,352583,290437,110543
7,Apple,AAPL,2024,2024-09-28,391035,93736,364980,308030,118254
8,Apple,AAPL,2025,2025-09-27,416161,112010,359241,285508,111482


---
## 4. Calculate Year-over-Year Growth Rates

In [3]:
# Calculate year-over-year growth rates for Total Revenue and Net Income
df['Revenue Growth (%)'] = df.groupby('Company Name')['Total Revenue'].pct_change() * 100
df['Net Income Growth (%)'] = df.groupby('Company Name')['Net Income'].pct_change() * 100

# Fill NA values that result from pct_change calculations with 0 or an appropriate value
df.fillna(0, inplace=True)

# Display the dataframe to verify the calculations
print(df.to_string())

  Company Name Ticker  Fiscal Year Fiscal Year End  Total Revenue  Net Income  Total Assets  Total Liabilities  Cash Flow from Operating Activities  Revenue Growth (%)  Net Income Growth (%)
0    Microsoft   MSFT         2023      2023-06-30         211915       72361        411976             205753                                87582            0.000000               0.000000
1    Microsoft   MSFT         2024      2024-06-30         245122       88136        512163             243686                               118548           15.669962              21.800417
2    Microsoft   MSFT         2025      2025-06-30         281724      101832        619003             275524                               136162           14.932156              15.539621
3        Tesla   TSLA         2023      2023-12-31          96773       14974        106618              43009                                13256            0.000000               0.000000
4        Tesla   TSLA         2024      2024-

In [4]:
# Optionally, you could summarize these findings for each company
summary = df.groupby('Company Name').agg({
    'Revenue Growth (%)': 'mean',
    'Net Income Growth (%)': 'mean'
}).reset_index()

print("\nYear-over-Year Average Growth Rates (%):")
print(summary)


Year-over-Year Average Growth Rates (%):
  Company Name  Revenue Growth (%)  Net Income Growth (%)
0        Apple            2.815835               5.378404
1    Microsoft           10.200706              12.446679
2        Tesla           -0.661040             -32.779021


---
## 5. Additional Financial Metrics

In [5]:
# Profit Margin and Liabilities/Assets ratio
df['Profit Margin (%)'] = (df['Net Income'] / df['Total Revenue']) * 100
df['Liabilities / Assets (%)'] = (df['Total Liabilities'] / df['Total Assets']) * 100
df['Operating Cash Flow Growth (%)'] = df.groupby('Company Name')['Cash Flow from Operating Activities'].pct_change() * 100
df['Operating Cash Flow Growth (%)'].fillna(0, inplace=True)

print(df[['Company Name','Fiscal Year','Profit Margin (%)','Liabilities / Assets (%)','Operating Cash Flow Growth (%)']].round(2).to_string())

  Company Name  Fiscal Year  Profit Margin (%)  Liabilities / Assets (%)  Operating Cash Flow Growth (%)
0    Microsoft         2023              34.15                     49.94                             NaN
1    Microsoft         2024              35.96                     47.58                           35.36
2    Microsoft         2025              36.15                     44.51                           14.86
3        Tesla         2023              15.47                     40.34                             NaN
4        Tesla         2024               7.32                     39.64                           12.58
5        Tesla         2025               4.07                     39.87                           -1.18
6        Apple         2023              25.31                     82.37                             NaN
7        Apple         2024              23.97                     84.40                            6.98
8        Apple         2025              26.92         

/tmp/ipykernel_637/3892063992.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Operating Cash Flow Growth (%)'].fillna(0, inplace=True)


---
## 6. Summary by Company

In [6]:
full_summary = df.groupby('Company Name').agg(
    Avg_Revenue_Growth=('Revenue Growth (%)', 'mean'),
    Avg_NetIncome_Growth=('Net Income Growth (%)', 'mean'),
    Avg_Profit_Margin=('Profit Margin (%)', 'mean'),
    Avg_Liabilities_Assets=('Liabilities / Assets (%)', 'mean'),
    Avg_OCF_Growth=('Operating Cash Flow Growth (%)', 'mean')
).round(2).reset_index()

print("=== 3-Year Average Financial Summary ===")
print(full_summary.to_string())

=== 3-Year Average Financial Summary ===
  Company Name  Avg_Revenue_Growth  Avg_NetIncome_Growth  Avg_Profit_Margin  Avg_Liabilities_Assets  Avg_OCF_Growth
0        Apple                2.82                  5.38              25.40                   82.08            0.62
1    Microsoft               10.20                 12.45              35.42                   47.34           25.11
2        Tesla               -0.66                -32.78               8.95                   39.95            5.70


---
## 7. Key Findings & Insights

### Microsoft (MSFT)
- Revenue grew from $211.9B → $281.7B — strongest growth driven by Azure cloud and AI/Copilot services
- Net income rose consistently from $72.4B → $101.8B
- **Highest profit margin (~36%)** — reflects high-margin SaaS/cloud business model
- Debt-to-asset ratio improving (49.9% → 44.5%) — strengthening balance sheet
- **MD&A insight**: AI integration across product lines (Copilot, Azure OpenAI) primary growth driver

### Tesla (TSLA)
- Revenue relatively flat ($96.8B → $94.8B) — pricing pressure and EV competition
- **Net income declined sharply**: $14.9B → $3.9B — aggressive price cuts compressed margins significantly
- **Lowest profit margin (~4%)** — capital-intensive manufacturing model
- ⚠️ **Key anomaly for AI chatbot**: Declining profitability despite stable revenue needs contextual explanation
- **MD&A insight**: Management cited vehicle price reductions as strategic market share defense

### Apple (AAPL)
- **Largest revenue base** ($383B → $416B) — stable growth
- Net income recovered in 2025 ($112B) after dip in 2024
- **Best operating cash flow** — exceeded $110B all three years
- High liabilities/assets ratio (~79-84%) due to strategic share buyback program — not financial distress
- **MD&A insight**: Services segment (App Store, iCloud, Apple Pay) driving margin improvement

---
## 8. Recommendations for AI Chatbot (GFC)

1. **Priority metrics**: Revenue Growth %, Net Profit Margin, Liabilities/Assets Ratio, OCF Growth
2. **Include MD&A context** in chatbot training — qualitative explanations essential for accurate interpretation
3. **Anomaly detection**: Flag unusual patterns (Tesla's profit decline) with contextual explanation
4. **GAAP vs non-GAAP**: Chatbot must clearly distinguish between the two
5. **Quarterly refresh**: Integrate 10-Q data for real-time insights between annual reports
6. **Risk factor awareness**: Surface key risks (Item 1A) alongside financial metrics

In [7]:
# Export final processed dataset
df.to_csv('financial_data_processed.csv', index=False)
print("Processed dataset exported successfully!")
print("Final shape:", df.shape)
print("Columns:", list(df.columns))

Processed dataset exported successfully!
Final shape: (9, 14)
Columns: ['Company Name', 'Ticker', 'Fiscal Year', 'Fiscal Year End', 'Total Revenue', 'Net Income', 'Total Assets', 'Total Liabilities', 'Cash Flow from Operating Activities', 'Revenue Growth (%)', 'Net Income Growth (%)', 'Profit Margin (%)', 'Liabilities / Assets (%)', 'Operating Cash Flow Growth (%)']


---
*Prepared by: Harsh | BCG GenAI Consulting Team*  
*Submitted to: Aisha, Senior Data Scientist*  
*Data Source: SEC EDGAR 10-K Filings*  
*Project: GFC AI-Powered Financial Chatbot – Task 1*